In [173]:
import re
import pickle
import nltk 
import sklearn
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics import roc_auc_score

In [133]:
%reload_ext watermark
%watermark -a "Matheus dos Anjos" --iversions

Author: Matheus dos Anjos

re     : 2.2.1
numpy  : 1.26.4
sklearn: 1.5.1
pandas : 2.2.2
nltk   : 3.9.1



In [134]:
df = pd.read_csv('C:/Users/conta/OneDrive/Desktop/Projetos-/Projetos Python/Projetos de Machine Learning/dados/sentimento.csv')

In [135]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [136]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [137]:
df.shape

(50000, 2)

In [138]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [139]:
df.sentiment.replace('positive', 1, inplace = True)
df.sentiment.replace('negative', 0, inplace = True)

C:\Users\conta\AppData\Local\Temp\ipykernel_7112\2993723773.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df.sentiment.replace('positive', 1, inplace = True)
C:\Users\conta\AppData\Local\Temp\ipykernel_7112\2993723773.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For ex

In [140]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [141]:
df.review[0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [142]:
def limpa_dados(texto):
    cleaned = re.compile(r'<.*>')
    return re.sub(cleaned, '', texto)

In [143]:
texto_com_tags = '<p>EXEMPLO<p>'
texto_limpo = limpa_dados(texto_com_tags)
print(texto_limpo)

In [144]:
df.review = df.review.apply(limpa_dados)

In [145]:
df.review[0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.I would say the main appeal of the show is due to the fact that it goes where other shows wouldn't dare. Forget pretty pictures painted for mainstream audiences, forget charm, forget romance...OZ doesn't mess around. The first episode I ever saw struck me as so nasty it was surreal, I couldn't say I was ready for it, but as I watched more, I developed a taste for Oz, and got accustomed to the high levels of graphic violence. Not just violence, but injustice (crooked guards who'll be sold out for a nickel, inmates who'll kill on order and get away with it, well mannered, middle class inmates being turned into prison bitches due to their lack of street skills or prison experience) Watching Oz, you may become comfortable with what is uncomfortable viewing....thats if you can get in touch with your darker side."

In [146]:
def limpa_caracter_especial(texto):
    rem = ''
    for i in texto:
        if i.isalnum():
            rem = rem + i
        else:
            rem = rem + ' '
    return rem

In [147]:
df.review = df.review.apply(limpa_caracter_especial)

In [148]:
df.review[0]

'One of the other reviewers has mentioned that after watching just 1 Oz episode you ll be hooked  They are right  as this is exactly what happened with me I would say the main appeal of the show is due to the fact that it goes where other shows wouldn t dare  Forget pretty pictures painted for mainstream audiences  forget charm  forget romance   OZ doesn t mess around  The first episode I ever saw struck me as so nasty it was surreal  I couldn t say I was ready for it  but as I watched more  I developed a taste for Oz  and got accustomed to the high levels of graphic violence  Not just violence  but injustice  crooked guards who ll be sold out for a nickel  inmates who ll kill on order and get away with it  well mannered  middle class inmates being turned into prison bitches due to their lack of street skills or prison experience  Watching Oz  you may become comfortable with what is uncomfortable viewing    thats if you can get in touch with your darker side '

In [149]:
def converter_minusculo(texto):
    return texto.lower()

In [150]:
df.review = df.review.apply(converter_minusculo)

In [151]:
df.review[0]

'one of the other reviewers has mentioned that after watching just 1 oz episode you ll be hooked  they are right  as this is exactly what happened with me i would say the main appeal of the show is due to the fact that it goes where other shows wouldn t dare  forget pretty pictures painted for mainstream audiences  forget charm  forget romance   oz doesn t mess around  the first episode i ever saw struck me as so nasty it was surreal  i couldn t say i was ready for it  but as i watched more  i developed a taste for oz  and got accustomed to the high levels of graphic violence  not just violence  but injustice  crooked guards who ll be sold out for a nickel  inmates who ll kill on order and get away with it  well mannered  middle class inmates being turned into prison bitches due to their lack of street skills or prison experience  watching oz  you may become comfortable with what is uncomfortable viewing    thats if you can get in touch with your darker side '

In [152]:
def remove_stopwords_sklearn(texto):
    words = texto.split()
    return ' '.join([w for w in words if w not in ENGLISH_STOP_WORDS])

In [153]:
df.review = df.review.apply(remove_stopwords_sklearn)

In [154]:
df.review[0]

'reviewers mentioned watching just 1 oz episode ll hooked right exactly happened say main appeal fact goes shows wouldn t dare forget pretty pictures painted mainstream audiences forget charm forget romance oz doesn t mess episode saw struck nasty surreal couldn t say ready watched developed taste oz got accustomed high levels graphic violence just violence injustice crooked guards ll sold nickel inmates ll kill order away mannered middle class inmates turned prison bitches lack street skills prison experience watching oz comfortable uncomfortable viewing thats touch darker'

In [155]:
def stemmer(texto):
    objeto_stemmer = SnowballStemmer('english')
    return " ".join([objeto_stemmer.stem(w) for w in texto])

In [156]:
df.review = df.review.apply(stemmer)

In [157]:
df.review[0]

'r e v i e w e r s   m e n t i o n e d   w a t c h i n g   j u s t   1   o z   e p i s o d e   l l   h o o k e d   r i g h t   e x a c t l y   h a p p e n e d   s a y   m a i n   a p p e a l   f a c t   g o e s   s h o w s   w o u l d n   t   d a r e   f o r g e t   p r e t t y   p i c t u r e s   p a i n t e d   m a i n s t r e a m   a u d i e n c e s   f o r g e t   c h a r m   f o r g e t   r o m a n c e   o z   d o e s n   t   m e s s   e p i s o d e   s a w   s t r u c k   n a s t y   s u r r e a l   c o u l d n   t   s a y   r e a d y   w a t c h e d   d e v e l o p e d   t a s t e   o z   g o t   a c c u s t o m e d   h i g h   l e v e l s   g r a p h i c   v i o l e n c e   j u s t   v i o l e n c e   i n j u s t i c e   c r o o k e d   g u a r d s   l l   s o l d   n i c k e l   i n m a t e s   l l   k i l l   o r d e r   a w a y   m a n n e r e d   m i d d l e   c l a s s   i n m a t e s   t u r n e d   p r i s o n   b i t c h e s   l a c k   s t r e e t   s k i l l s   p r i

In [158]:
x = np.array(df.iloc[:, 0].values)

In [159]:
y = np.array(df.sentiment.values)

In [160]:
x_treino, x_teste, y_treino, y_teste = train_test_split(x, y, test_size = 0.2, random_state = 0)

In [161]:
type(x_treino)

numpy.ndarray

In [162]:
vetorizador = CountVectorizer(
    max_features=1000,
    min_df=2,  
    stop_words=None,  
    token_pattern=r'\b\w+\b'  
)

In [163]:
x_treino_final = vetorizador.fit_transform(x_treino).toarray()

In [164]:
x_teste_final = vetorizador.transform(x_teste).toarray()

In [165]:
print('x_treino_final:', x_treino_final.shape)
print('y_treino:', x_treino.shape)
print('x_treino_final:', x_teste_final.shape)
print('y_treino:', x_teste.shape)

x_treino_final: (40000, 67)
y_treino: (40000,)
x_treino_final: (10000, 67)
y_treino: (10000,)


In [166]:
modelo_v1 = GaussianNB()
modelo_v1.fit(x_treino_final, y_treino)

GaussianNB()

In [167]:
modelo_v2 = MultinomialNB(alpha = 1.0, fit_prior = True)
modelo_v2.fit(x_treino_final, y_treino)

MultinomialNB()

In [168]:
modelo_v3 = BernoulliNB(alpha = 1.0, fit_prior = True)
modelo_v3.fit(x_treino_final, y_treino) 

BernoulliNB()

In [169]:
ypred_v1 = modelo_v1.predict(x_teste_final)

In [170]:
ypred_v2 = modelo_v2.predict(x_teste_final)

In [171]:
ypred_v3 = modelo_v3.predict(x_teste_final)

In [172]:
print('Acuracia Modelo GaussianNB:', accuracy_score(y_teste, ypred_v1) * 100)
print('Acuracia Modelo MultinominalNB:', accuracy_score(y_teste, ypred_v2) * 100)
print('Acuracia Modelo BernoulliNB:', accuracy_score(y_teste, ypred_v3) * 100)

Acuracia Modelo GaussianNB: 51.31
Acuracia Modelo MultinominalNB: 60.78
Acuracia Modelo BernoulliNB: 55.900000000000006


In [174]:
y_proba = modelo_v1.predict_proba(x_teste_final)[:, 1]
auc = roc_auc_score(y_teste, y_proba)
print('AUC do modelo GaussianNB:', auc)

AUC do modelo GaussianNB: 0.5508685125571153


In [177]:
y_proba = modelo_v2.predict_proba(x_teste_final)[:, 1]
auc = roc_auc_score(y_teste, y_proba)
print('AUC do modelo MultinominalNB:', auc)

AUC do modelo MultinominalNB: 0.6427763160394859


In [178]:
y_proba = modelo_v3.predict_proba(x_teste_final)[:, 1]
auc = roc_auc_score(y_teste, y_proba)
print('AUC do modelo BernoulliNB:', auc)

AUC do modelo BernoulliNB: 0.5812241599838393


In [179]:
with open('modelo_v2.pkl', 'wb') as arquivo:
    pickle.dump(modelo_v2, arquivo)

In [183]:
with open('modelo_v2.pkl', 'rb') as arquivo:
    modelo_final = pickle.load(arquivo)

In [184]:
texto_novo = """This is probably the fastest-paced and most action-packed of the German Edgar Wallace ""krimi"" 
series, a cross between the Dr. Mabuse films of yore and 60's pop thrillers like Batman and the Man 
from UNCLE. It reintroduces the outrageous villain from an earlier film who dons a stylish monk's habit and 
breaks the necks of victims with the curl of a deadly whip. Set at a posh girls' school filled with lecherous 
middle-aged professors, and with the cops fondling their hot-to-trot secretaries at every opportunity, it 
certainly is a throwback to those wonderfully politically-incorrect times. There's a definite link to a later 
Wallace-based film, the excellent giallo ""Whatever Happened to Solange?"", which also concerns female students 
being corrupted by (and corrupting?) their elders. Quite appropriate to the monk theme, the master-mind villain 
uses booby-trapped bibles here to deal some of the death blows, and also maintains a reptile-replete dungeon 
to amuse his captive audiences. <br /><br />Alfred Vohrer was always the most playful and visually flamboyant 
of the series directors, and here the lurid colour cinematography is the real star of the show. The Monk appears 
in a raving scarlet cowl and robe, tastefully setting off the lustrous white whip, while appearing against 
purplish-night backgrounds. There's also a voyeur-friendly turquoise swimming pool which looks great both 
as a glowing milieu for the nubile students and as a shadowy backdrop for one of the murder scenes. 
The trademark ""kicker"" of hiding the ""Ende"" card somewhere in the set of the last scene is also quite 
memorable here. And there's a fine brassy and twangy score for retro-music fans.<br /><br />Fans of the series 
will definitely miss the flippant Eddie Arent character in these later films. Instead, the chief inspector 
Sir John takes on the role of buffoon, convinced that he has mastered criminal psychology after taking a few 
night courses. Unfortunately, Klaus Kinski had also gone on to bigger and better things. The krimis had 
lost some of their offbeat subversive charm by this point, and now worked on a much more blatant pop-culture 
level, which will make this one quite accessible to uninitiated viewers."""

In [185]:
tarefa1 = limpa_dados(texto_novo)
tarefa2 = limpa_caracter_especial(tarefa1)
tarefa3 = converter_minusculo(tarefa2)
tarefa4 = remove_stopwords_sklearn(tarefa3)
tarefa5 = stemmer(tarefa4)

In [186]:
print(tarefa5)

p r o b a b l y   f a s t e s t   p a c e d   a c t i o n   p a c k e d   g e r m a n   e d g a r   w a l l a c e   k r i m i   s e r i e s   c r o s s   d r   m a b u s e   f i l m s   y o r e   6 0   s   p o p   t h r i l l e r s   l i k e   b a t m a n   m a n   u n c l e   r e i n t r o d u c e s   o u t r a g e o u s   v i l l a i n   e a r l i e r   f i l m   d o n s   s t y l i s h   m o n k   s   h a b i t   b r e a k s   n e c k s   v i c t i m s   c u r l   d e a d l y   w h i p   s e t   p o s h   g i r l s   s c h o o l   f i l l e d   l e c h e r o u s   m i d d l e   a g e d   p r o f e s s o r s   c o p s   f o n d l i n g   h o t   t r o t   s e c r e t a r i e s   o p p o r t u n i t y   c e r t a i n l y   t h r o w b a c k   w o n d e r f u l l y   p o l i t i c a l l y   i n c o r r e c t   t i m e s   s   d e f i n i t e   l i n k   l a t e r   w a l l a c e   b a s e d   f i l m   e x c e l l e n t   g i a l l o   h a p p e n e d   s o l a n g e   c o n c e r n s 

In [187]:
type(tarefa5)

str

In [ ]:
tarefa5_array = np.array(tarefa5)

In [189]:
type(tarefa5_array)

numpy.ndarray

In [190]:
aval_final = vetorizador.transform(np.array([tarefa5_array])).toarray()

In [191]:
type(aval_final)

numpy.ndarray

In [194]:
previsao = modelo_final.predict(aval_final)

In [195]:
if previsao == 1:
    print("O Texto Indica Sentimento Positivo!")
else:
    print("O Texto Indica Sentimento Negativo!")

O Texto Indica Sentimento Negativo!
